# Full stage 5 — resume structural chunks or run CUDA semantic chunks
For the current local run, keep `STRATEGY='structure'`. It uses parallel CPU workers and exact tokenizer caching, including inside a GPU runtime. CUDA does not accelerate this tokenizer/algorithm. Changing to `semantic` runs BGE-M3 on GPU but creates a different experiment and cannot reuse structural chunks.
Read `docs/CHUNKING_STAGE5_COLAB_FULL.md`. No cells or tests have been executed by the assistant.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, shutil, subprocess, sys, zipfile, json
BUNDLE = Path('/content/drive/MyDrive/ViBioMIR/stage5_full_colab.zip')  # edit
STRATEGY = 'structure'  # fixed / structure / semantic
PROJECT = Path('/content/vibiomir_stage5_full')
RUN_DIR = Path('/content') / ('stage5_full_' + STRATEGY)
BACKUP_DIR = Path('/content/drive/MyDrive/ViBioMIR') / ('stage5_full_' + STRATEGY)
WORKERS = min(4, os.cpu_count() or 1)
ENCODER_BATCH_SIZE = 8
STOP_AFTER_BATCHES = 2  # timing/resume pilot; set None for the full continuation
assert STRATEGY in {'fixed', 'structure', 'semantic'}
assert BUNDLE.is_file(), f'Upload the full bundle first: {BUNDLE}'
PROJECT.mkdir(parents=True, exist_ok=True)
print({'strategy': STRATEGY, 'cpu_workers': WORKERS, 'local_output': str(RUN_DIR), 'drive_backup': str(BACKUP_DIR)})

In [ ]:
# Extract once per fresh runtime; computation and locks stay off Drive.
with zipfile.ZipFile(BUNDLE) as archive:
    required = sum(info.file_size for info in archive.infolist())
    assert shutil.disk_usage('/content').free > required * 2, 'Need local space for inputs, restored checkpoint and new output.'
    for info in archive.infolist():
        target = (PROJECT / info.filename).resolve()
        assert PROJECT.resolve() in target.parents, 'Unsafe ZIP entry'
    archive.extractall(PROJECT)
os.chdir(PROJECT)
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
requirements = 'requirements-stage5-colab.txt' if STRATEGY == 'semantic' else 'requirements-stage5.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', requirements], check=True)
print('If Colab requests a restart, restart then rerun the setup cells before proceeding.')

In [ ]:
input_manifest = json.loads((PROJECT / 'inputs/manifest.json').read_text())
assert input_manifest['state'] == 'complete', 'Input preparation is not complete.'
print('Prepared documents:', input_manifest['documents'], 'Input batches:', input_manifest['batches'])
SEED = PROJECT / 'checkpoint'
USE_SEED = False
if (SEED / 'manifest.json').is_file():
    seed_manifest = json.loads((SEED / 'manifest.json').read_text())
    USE_SEED = seed_manifest['identity']['strategy'] == STRATEGY
    print('Bundled checkpoint strategy:', seed_manifest['identity']['strategy'], 'Reuse:', USE_SEED)
if STRATEGY == 'semantic':
    import torch
    assert torch.cuda.is_available(), 'Select a CUDA GPU runtime for semantic chunks.'
    print('CUDA encoder:', torch.cuda.get_device_name(0))
else:
    print('GPU use: 0. This strategy uses Colab CPU workers; a larger GPU does not speed up tokenization.')

In [ ]:
# Manually execute small offline + real-tokenizer checks before a full continuation.
test_env = dict(os.environ, VIBIOMIR_TOKENIZER_DIR=str(PROJECT / 'tokenizer'))
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_stage5*.py', '-v'], env=test_env, check=True)

In [ ]:
# Use an importable CLI subprocess so spawned workers also work in Colab.
command = [sys.executable, '-m', 'vibiomir.chunk_parallel',
    '--input-dir', str(PROJECT / 'inputs'), '--output-dir', str(RUN_DIR),
    '--tokenizer-dir', str(PROJECT / 'tokenizer'), '--config', str(PROJECT / 'configs/chunking_stage5.json'),
    '--strategy', STRATEGY, '--workers', str(WORKERS),
    '--encoder-batch-size', str(ENCODER_BATCH_SIZE), '--backup-dir', str(BACKUP_DIR), '--resume']
if USE_SEED:
    command += ['--seed-checkpoint-dir', str(SEED)]
if STOP_AFTER_BATCHES is not None:
    command += ['--stop-after-batches', str(STOP_AFTER_BATCHES)]
import time
started = time.monotonic()
subprocess.run(command, check=True)
print('Runner invocation seconds (including restore/startup/backup):', round(time.monotonic() - started, 2))
result = json.loads((RUN_DIR / 'manifest.json').read_text())
print('State:', result['state'])
if result['state'] != 'complete':
    print('Review batch timings. Set STOP_AFTER_BATCHES=None and rerun this cell to continue. No preparation rerun is needed.')

In [ ]:
result = json.loads((RUN_DIR / 'manifest.json').read_text())
if result['state'] == 'complete':
    subprocess.run([sys.executable, '-m', 'vibiomir', 'validate-chunks',
        '--input-dir', str(PROJECT / 'inputs'), '--output-dir', str(RUN_DIR),
        '--tokenizer-dir', str(PROJECT / 'tokenizer')], check=True)
    from vibiomir.chunk_colab import save_checkpoint
    save_checkpoint(RUN_DIR, BACKUP_DIR)
    print('Completed output and validation saved:', BACKUP_DIR)
else:
    print('Validation waits for a complete output; continue the run cell first.')